# 05 — Feature Extraction

**Goal:** Extract physiological features from 1-minute PPG windows and align them with blood pressure labels.

Each window yields **356 features** grouped into 7 categories:

| # | Feature group | What it measures |
|---|---------------|------------------|
| 1 | **Area** | Area under each PPG cycle (blood vessel compliance) |
| 2 | **Width–Amplitude** | Amplitude at 25 / 50 / 75% of peak height (vascular resistance) |
| 3 | **ROR / ROF** | Rate of Rise and Rate of Fall (systolic / diastolic timing) |
| 4 | **Cycle Time** | Min-to-min interval (heart rate derived) |
| 5 | **Y Height** | Raw systolic peak amplitude |
| 6 | **STT** | Slope-to-Amplitude ratio (ROR / Y) |
| 7 | **TV** | Total Variation of the filtered signal |

**Six BP targets are predicted:**
`MAP`, `ΔMAP`, `SBP`, `ΔSBP`, `DBP`, `ΔDBP`

**Inputs:** `data/processed/ml_train.pkl`, `data/processed/ml_test.pkl`

**Outputs:** `data/processed/features_train.pkl`, `data/processed/features_test.pkl`

---
### Pipeline position
```
01 Import → 02 Signal Processing → 03 Interpolation → 04 ML Data Loading → [05 Features] → 06 Models
```

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import signal as sp_signal

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ROOT     = Path.cwd().parent if Path.cwd().name == 'improved' else Path.cwd().parent.parent
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'

SAMPLE_RATE    = 100     # Hz
WINDOW_SECONDS = 60      # 1-minute windows for feature extraction
WINDOW_SAMPLES = SAMPLE_RATE * WINDOW_SECONDS   # 6000 samples

# Amplitude percentages for Width-Amplitude features
AMPLITUDE_FRACTIONS = [0.25, 0.50, 0.75]

In [ ]:
with open(PROCESSED_DIR / 'ml_train.pkl', 'rb') as f:
    ml_train = pickle.load(f)
with open(PROCESSED_DIR / 'ml_test.pkl', 'rb') as f:
    ml_test  = pickle.load(f)

bp_train  = ml_train['bp'];   ppg_train = ml_train['ppg']
bp_test   = ml_test['bp'];    ppg_test  = ml_test['ppg']
train_ids = ml_train['patient_ids']
test_ids  = ml_test['patient_ids']

print(f'Train: {len(train_ids)} patients,  Test: {len(test_ids)} patients')

## 1. Signal Filtering

Before extracting features, each 1-minute window is bandpass-filtered (0.5–8 Hz) to isolate the cardiac frequency band.

In [ ]:
def bandpass_filter(data: np.ndarray,
                    lowcut: float = 0.5, highcut: float = 8.0,
                    fs: float = SAMPLE_RATE, order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth bandpass filter (0.5–8 Hz)."""
    nyq = fs / 2
    b, a = sp_signal.butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return sp_signal.filtfilt(b, a, data)

## 2. Cycle Detection

Cardiac cycles are identified as min → max → min triplets in the filtered signal.
Only cycles containing exactly one systolic peak are counted as valid.

In [ ]:
def detect_cycles(signal: np.ndarray, min_distance_samples: int = 30) -> list:
    """
    Detect valid cardiac cycles in a PPG signal.

    Returns a list of (min_start, max_peak, min_end) index tuples.
    Each tuple represents one systolic–diastolic cycle.
    """
    maxima, _ = sp_signal.find_peaks(signal,  distance=min_distance_samples)
    minima, _ = sp_signal.find_peaks(-signal, distance=min_distance_samples)

    cycles = []
    for i in range(len(minima) - 1):
        peaks_between = maxima[(maxima > minima[i]) & (maxima < minima[i + 1])]
        if len(peaks_between) == 1:
            cycles.append((minima[i], peaks_between[0], minima[i + 1]))
    return cycles

## 3. Feature Extraction Functions

Each function operates on a list of detected cycles and the filtered PPG signal.

In [ ]:
def feature_area(signal: np.ndarray, cycles: list) -> np.ndarray:
    """
    Area under the PPG curve for each cycle (trapezoidal integration).
    Reflects blood vessel compliance and stroke volume.
    """
    return np.array([np.trapz(signal[c[0]:c[2]]) for c in cycles])


def feature_width_amplitude(signal: np.ndarray, cycles: list,
                             fractions: list = AMPLITUDE_FRACTIONS) -> np.ndarray:
    """
    Signal width at a given fraction of the peak amplitude.

    For each fraction in *fractions* and each cycle, count the number of
    samples above the threshold: trough + fraction * (peak - trough).
    High width-amplitude at 25% → wider pulse → lower vascular resistance.
    """
    results = []
    for frac in fractions:
        widths = []
        for min_s, max_p, min_e in cycles:
            trough = signal[min_s]
            peak   = signal[max_p]
            thresh = trough + frac * (peak - trough)
            widths.append(np.sum(signal[min_s:min_e] >= thresh))
        results.append(np.array(widths))
    return np.column_stack(results) if results else np.empty((len(cycles), 0))


def feature_ror_rof(signal: np.ndarray, cycles: list) -> tuple:
    """
    Rate of Rise (ROR) and Rate of Fall (ROF) for each cycle.

    ROR = (peak - trough_before) / (peak_idx - trough_idx)   [amplitude / sample]
    ROF = (peak - trough_after)  / (trough_after_idx - peak_idx)
    """
    ror, rof = [], []
    for min_s, max_p, min_e in cycles:
        dt_rise = max_p - min_s
        dt_fall = min_e - max_p
        ror.append((signal[max_p] - signal[min_s]) / dt_rise if dt_rise else np.nan)
        rof.append((signal[max_p] - signal[min_e]) / dt_fall if dt_fall else np.nan)
    return np.array(ror), np.array(rof)


def feature_cycle_time(cycles: list) -> np.ndarray:
    """Duration (samples) between consecutive diastolic troughs."""
    return np.array([c[2] - c[0] for c in cycles], dtype=float)


def feature_y_height(signal: np.ndarray, cycles: list) -> np.ndarray:
    """Systolic peak amplitude (raw signal value at the peak)."""
    return np.array([signal[c[1]] for c in cycles])


def feature_stt(signal: np.ndarray, cycles: list) -> np.ndarray:
    """
    Slope-to-Amplitude ratio (STT) = ROR / Y_height.
    Normalises rise rate by peak height; captures timing independently of amplitude.
    """
    ror, _ = feature_ror_rof(signal, cycles)
    y = feature_y_height(signal, cycles)
    return np.where(y != 0, ror / y, np.nan)


def feature_total_variation(signal: np.ndarray) -> float:
    """Total variation = sum of absolute first differences. Measures signal roughness."""
    return float(np.sum(np.abs(np.diff(signal))))

## 4. Aggregate Features per Window

For each feature, the per-cycle values are summarised with mean, std, min, and max → giving 4 statistical descriptors per feature.

In [ ]:
def summarise(arr: np.ndarray) -> dict:
    """Return mean, std, min, max of a 1-D array as a flat dict."""
    arr = arr[~np.isnan(arr)] if arr.ndim == 1 else arr
    return {
        'mean': np.nanmean(arr),
        'std':  np.nanstd(arr),
        'min':  np.nanmin(arr),
        'max':  np.nanmax(arr),
    }


def extract_features(signal: np.ndarray) -> dict:
    """
    Extract all feature groups from a single PPG window.

    Parameters
    ----------
    signal : 1-D array of raw (unfiltered) PPG values for one window

    Returns
    -------
    Flat dict of feature_name -> value
    """
    filtered = bandpass_filter(signal)
    cycles   = detect_cycles(filtered)

    if len(cycles) < 3:
        return {}   # not enough cycles for reliable features

    features = {}

    # 1. Area
    for stat, val in summarise(feature_area(filtered, cycles)).items():
        features[f'area_{stat}'] = val

    # 2. Width-Amplitude (per fraction)
    wa = feature_width_amplitude(filtered, cycles)
    for fi, frac in enumerate(AMPLITUDE_FRACTIONS):
        frac_label = int(frac * 100)
        for stat, val in summarise(wa[:, fi]).items():
            features[f'width_amp_{frac_label}pct_{stat}'] = val

    # 3. ROR / ROF
    ror, rof = feature_ror_rof(filtered, cycles)
    for stat, val in summarise(ror).items():
        features[f'ror_{stat}'] = val
    for stat, val in summarise(rof).items():
        features[f'rof_{stat}'] = val

    # 4. Cycle Time
    for stat, val in summarise(feature_cycle_time(cycles)).items():
        features[f'cycle_time_{stat}'] = val

    # 5. Y Height
    for stat, val in summarise(feature_y_height(filtered, cycles)).items():
        features[f'y_height_{stat}'] = val

    # 6. STT
    for stat, val in summarise(feature_stt(filtered, cycles)).items():
        features[f'stt_{stat}'] = val

    # 7. Total Variation
    features['total_variation'] = feature_total_variation(filtered)

    return features

## 5. Build Feature DataFrames

Slide a 1-minute window over each patient's PPG recording and pair each window with the simultaneous blood pressure measurement.

In [ ]:
# BP label columns — update these names to match your actual DataFrame columns
BP_LABEL_COLS = {
    'MAP':  'L1_MAP',
    'DMAP': 'L2_DMAP',
    'SBP':  'L3_SBP',
    'DSBP': 'L4_DSBP',
    'DBP':  'L5_DBP',
    'DDBP': 'L6_DDBP',
}


def build_feature_df(ppg_dict: dict, bp_dict: dict, patient_ids: list) -> pd.DataFrame:
    """
    Extract features and BP labels for all patients in *patient_ids*.

    For each patient, non-overlapping 1-minute windows are extracted from the
    PPG recording. The corresponding BP label is taken as the mean BP value
    within that window's time range.

    Returns a DataFrame where each row is one window,
    with columns: patient_id, all feature columns, all label columns.
    """
    rows = []
    for pid in patient_ids:
        if pid not in ppg_dict or pid not in bp_dict:
            continue
        ppg_df = ppg_dict[pid]
        bp_df  = bp_dict[pid]
        ppg_signal = ppg_df['PLETH'].values

        # Determine available BP label columns
        label_cols = {k: v for k, v in BP_LABEL_COLS.items() if v in bp_df.columns}

        for start in range(0, len(ppg_signal) - WINDOW_SAMPLES + 1, WINDOW_SAMPLES):
            window = ppg_signal[start: start + WINDOW_SAMPLES]
            if np.isnan(window).mean() > 0.1:   # skip if > 10% of window is NaN
                continue
            window = window.copy()
            window[np.isnan(window)] = np.nanmedian(window)   # fill residual NaN

            feats = extract_features(window)
            if not feats:
                continue

            row = {'patient_id': pid, **feats}
            for label_name, col in label_cols.items():
                # Use mean BP over the same time window (approximate alignment)
                window_bp = bp_df[col].iloc[
                    int(start * len(bp_df) / len(ppg_signal)):
                    int((start + WINDOW_SAMPLES) * len(bp_df) / len(ppg_signal))
                ]
                row[label_name] = window_bp.mean()

            rows.append(row)

    return pd.DataFrame(rows)


print('Extracting training features...')
features_train = build_feature_df(ppg_train, bp_train, train_ids)
print(f'Train: {len(features_train)} windows,  {features_train.shape[1]} columns')

print('Extracting test features...')
features_test = build_feature_df(ppg_test, bp_test, test_ids)
print(f'Test : {len(features_test)} windows,  {features_test.shape[1]} columns')

## 6. Dataset Summary

In [ ]:
label_names = list(BP_LABEL_COLS.keys())
existing_labels = [l for l in label_names if l in features_train.columns]

if existing_labels:
    print('BP label statistics (train):')
    print(features_train[existing_labels].describe().round(2))

    fig, axes = plt.subplots(1, len(existing_labels), figsize=(4 * len(existing_labels), 3))
    if len(existing_labels) == 1:
        axes = [axes]
    for ax, col in zip(axes, existing_labels):
        ax.hist(features_train[col].dropna(), bins=40, color='steelblue', edgecolor='none')
        ax.set_title(col)
        ax.set_xlabel('mmHg')
    plt.suptitle('BP label distributions (train)', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No BP label columns found — check BP_LABEL_COLS mapping above.')

## 7. Save Feature DataFrames

In [ ]:
for filename, df in [
    ('features_train.pkl', features_train),
    ('features_test.pkl',  features_test),
]:
    with open(PROCESSED_DIR / filename, 'wb') as f:
        pickle.dump(df, f)
    print(f'Saved → data/processed/{filename}  ({len(df)} rows)')